In [ ]:
# --- Colab: Upload .conllu/.conllup -> Gemini sentence segmentation -> outputs ---
!pip -q install -U google-genai

import os, json, re
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple, Union

from google.colab import files, userdata
from google import genai
from google.genai import types


In [ ]:
# ----------------------------
# 0) CONFIG
# ----------------------------
MODEL_NAME        = "gemini-2.5-pro"
MAX_RETRIES       = 3      # retries per Gemini call
HARD_BOUNDARY_GAP = 2.0    # seconds — utterance gaps >= this are ALWAYS mandatory sentence
                            # boundaries; Gemini never sees tokens across such a gap


In [ ]:
# ----------------------------
# 1) AUTH (Colab Secrets)
# ----------------------------
api_key = userdata.get("GEMINI_API_KEY") or userdata.get("GOOGLE_API_KEY")
if not api_key:
    raise ValueError(
        "No API key found. In Colab, add a secret named GEMINI_API_KEY (or GOOGLE_API_KEY)."
    )

client = genai.Client(api_key=api_key)


In [ ]:

# ----------------------------
# 2) PARSE CONLLU/CONLLUP
# ----------------------------
@dataclass
class TokenEntry:
    cols: List[str]            # full columns of the line
    token: str                 # FORM
    meta: Dict[str, str]       # metadata from surrounding # key = value lines


def parse_conllu_like(path: str) -> Tuple[List[TokenEntry], List[str]]:
    """
    Returns:
      - token_entries (aligned with the token stream)
      - all_tokens (FORM column)
    Notes:
      - skips MWT lines (ID contains '-') and empty nodes (ID contains '.')
      - collects metadata from comment lines (# key = value) and attaches to tokens
      - metadata is carried forward until a blank line resets it
    """
    token_entries: List[TokenEntry] = []
    all_tokens: List[str] = []

    cur_meta: Dict[str, str] = {}
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.rstrip("\n")
            if not line:
                cur_meta = {}  # reset meta at original boundary
                continue

            if line.startswith("#"):
                m = re.match(r"#\s*([^=]+?)\s*=\s*(.*)\s*$", line)
                if m:
                    k = m.group(1).strip()
                    v = m.group(2).strip()
                    cur_meta[k] = v
                continue

            cols = line.split("\t")
            if len(cols) < 2:
                continue

            tok_id = cols[0]
            if "-" in tok_id or "." in tok_id:
                continue  # skip MWT / empty nodes

            form = cols[1]
            token_entries.append(TokenEntry(cols=cols, token=form, meta=dict(cur_meta)))
            all_tokens.append(form)

    return token_entries, all_tokens


In [ ]:
# ----------------------------
# 3) SPEAKER RUNS + PAUSE UTILITIES
# ----------------------------
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class UttBoundary:
    """Pause between two consecutive EXB utterances from the same speaker."""
    after_tok_idx: int    # 0-based index in speaker run; last token of prev utt
    gap_sec:       float  # next_start - prev_end in seconds
    is_hard:       bool   # True if gap >= HARD_BOUNDARY_GAP -> mandatory sentence boundary


def _safe_float(x: Optional[str]) -> Optional[float]:
    try:
        return float(x) if x is not None else None
    except (ValueError, TypeError):
        return None


def pause_marker(gap_sec: float) -> Optional[str]:
    """Middle-dot pause marker, or None for non-positive gaps."""
    if gap_sec <= 0:
        return None
    elif gap_sec < 0.5:
        return "\u00b7"            # \u00b7
    elif gap_sec < 1.0:
        return "\u00b7\u00b7"      # \u00b7\u00b7
    else:
        return "\u00b7\u00b7\u00b7"  # \u00b7\u00b7\u00b7


def build_speaker_runs(token_entries):
    """
    Split flat token list into consecutive same-speaker runs.
    Within each run, EXB utterance boundaries are detected by changes in
    (start_time, end_time) — values shared by all tokens of one EXB utterance.
    """
    if not token_entries:
        return []

    runs = []

    def flush(run_toks):
        if not run_toks:
            return
        speaker    = run_toks[0].meta.get("speaker_abbr", "_")
        boundaries = []
        prev_key   = (run_toks[0].meta.get("start_time"),
                      run_toks[0].meta.get("end_time"))

        for i in range(1, len(run_toks)):
            cur_key = (run_toks[i].meta.get("start_time"),
                       run_toks[i].meta.get("end_time"))
            if cur_key != prev_key:
                prev_end   = _safe_float(run_toks[i-1].meta.get("end_time"))   or 0.0
                next_start = _safe_float(run_toks[i].meta.get("start_time"))   or 0.0
                gap = next_start - prev_end
                boundaries.append(UttBoundary(
                    after_tok_idx = i - 1,
                    gap_sec       = gap,
                    is_hard       = gap >= HARD_BOUNDARY_GAP,
                ))
                prev_key = cur_key

        runs.append({"speaker": speaker, "tokens": run_toks, "boundaries": boundaries})

    current = []
    current_spk = None

    for te in token_entries:
        spk = te.meta.get("speaker_abbr", "_")
        if spk != current_spk:
            flush(current)
            current     = []
            current_spk = spk
        current.append(te)

    flush(current)
    return runs


def split_run_at_hard_boundaries(run):
    """
    Split a speaker run into sub-runs at every hard boundary (gap >= HARD_BOUNDARY_GAP).
    Each sub-run contains only the tokens and soft boundaries within that segment.
    Soft boundaries are re-indexed to be local (0-based) within the sub-run.

    Hard boundaries become natural sentence boundaries — Gemini never sees tokens
    across them, so no pause marker is inserted there.
    """
    tokens     = run["tokens"]
    boundaries = run["boundaries"]
    speaker    = run["speaker"]

    hard_cuts = sorted(b.after_tok_idx for b in boundaries if b.is_hard)

    if not hard_cuts:
        return [run]

    starts = [0]           + [c + 1 for c in hard_cuts]
    ends   = [c + 1 for c in hard_cuts] + [len(tokens)]

    sub_runs = []
    for seg_start, seg_end in zip(starts, ends):
        seg_toks = tokens[seg_start:seg_end]
        if not seg_toks:
            continue

        # Keep only soft boundaries strictly internal to this sub-run
        seg_bounds = [
            UttBoundary(
                after_tok_idx = b.after_tok_idx - seg_start,
                gap_sec       = b.gap_sec,
                is_hard       = False,
            )
            for b in boundaries
            if not b.is_hard
            and seg_start <= b.after_tok_idx
            and b.after_tok_idx + 1 < seg_end
        ]

        sub_runs.append({"speaker": speaker, "tokens": seg_toks, "boundaries": seg_bounds})

    return sub_runs


In [ ]:
# ----------------------------
# 4) GEMINI CALL  —  semantic utterance grouping
# ----------------------------
import time

def _extract_json(text):
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    for pat in [r"(\{.*\})", r"(\[.*\])"]:
        m = re.search(pat, text, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(1))
            except Exception:
                pass
    raise ValueError(f"Could not parse JSON:\n{text[:300]}")


def _build_utterances(sub_run):
    """
    Convert a sub-run into utterance dicts:
      id, text, tokens, gap_after_sec (None for last utterance)
    """
    tokens     = sub_run["tokens"]      # List[TokenEntry]
    boundaries = sub_run["boundaries"]  # List[UttBoundary], all soft

    if not boundaries:
        return [{"id": 0,
                 "text": " ".join(te.token for te in tokens),
                 "tokens": [te.token for te in tokens],
                 "gap_after_sec": None}]

    cuts   = [b.after_tok_idx for b in boundaries]
    starts = [0] + [c + 1 for c in cuts]
    ends   = [c + 1 for c in cuts] + [len(tokens)]

    utts = []
    for i, (s, e) in enumerate(zip(starts, ends)):
        toks = [te.token for te in tokens[s:e]]
        gap  = boundaries[i].gap_sec if i < len(boundaries) else None
        utts.append({"id": i, "text": " ".join(toks),
                     "tokens": toks, "gap_after_sec": gap})
    return utts


def segment_utterances(sub_run, model_name=MODEL_NAME):
    """
    Semantic sentence grouping via Gemini.

    Sends EXB utterances as named units (with gap durations) and asks Gemini
    to group consecutive ones into sentences.  Gemini reads and understands
    each utterance before deciding — it never operates on a raw token stream.

    Token preservation is guaranteed: we just concatenate utterance tokens per group.
    Pause markers (·/··/···) are inserted later by write_corrected_conllup.
    """
    utts       = _build_utterances(sub_run)
    all_tokens = [te.token for te in sub_run["tokens"]]

    if len(utts) == 1:
        return [all_tokens]

    n = len(utts)
    prompt = {
        "task": (
            "You are segmenting spoken Torlak/BCS dialect speech into sentences. "
            "The utterances below are consecutive speech segments from the SAME speaker, "
            "each separated by a short pause (gap_after_sec). "
            "Decide which consecutive utterances belong to the SAME sentence "
            "and which start a NEW sentence."
        ),
        "utterances": [
            {
                "id": u["id"],
                "text": u["text"],
                **({"gap_after_sec": round(u["gap_after_sec"], 2)}
                   if u["gap_after_sec"] is not None else {}),
            }
            for u in utts
        ],
        "instructions": [
            "Read and understand each utterance before deciding.",
            "Two utterances belong to the SAME sentence when the second one "
            "syntactically completes or continues the first "
            "(e.g. a subordinate clause, a complement, a conjoined VP sharing the subject).",
            "Start a NEW sentence when an utterance introduces a new topic or "
            "is grammatically complete on its own.",
            "Use gap_after_sec as a cue: a longer gap makes a sentence boundary more likely, "
            "but semantics and grammar take priority.",
            "Never split a clause mid-way (e.g. do not separate a verb from its complement).",
            f"Return ONLY valid JSON: {\"groups\": [[0], [1, 2], ...]}",
            f"Every ID from 0 to {n - 1} must appear in exactly one group, in ascending order.",
        ],
        "output_format": {"groups": [[0], [1, 2]]},
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.models.generate_content(
                model=model_name,
                contents=json.dumps(prompt, ensure_ascii=False),
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                ),
            )
            data   = _extract_json(resp.text)
            groups = data.get("groups", [])
            flat   = [uid for g in groups for uid in g]
            if flat == list(range(n)):
                return [
                    [tok for uid in group for tok in utts[uid]["tokens"]]
                    for group in groups
                ]
            prompt["instructions"].insert(
                0,
                f"RETRY {attempt}: groups must contain every ID 0..{n - 1} "
                "exactly once in ascending order. Fix it.",
            )
        except Exception as e:
            print(f"  attempt {attempt} error: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(5 * attempt)

    # Fallback: each utterance is its own sentence
    print("  Gemini grouping failed — treating each utterance as a sentence.")
    return [u["tokens"] for u in utts]


def segment_all_runs(runs, model_name=MODEL_NAME):
    """
    Segment every speaker run independently.

    Each run is first split at hard boundaries (gap >= HARD_BOUNDARY_GAP).
    Within each resulting sub-run, EXB utterances are sent to Gemini as named
    units with gap durations so it can understand each before grouping them
    into sentences.
    """
    sentences_per_run = []
    total = sum(len(r["tokens"]) for r in runs)
    done  = 0

    for ri, run in enumerate(runs):
        sub_runs = split_run_at_hard_boundaries(run)
        n_hard   = len(sub_runs) - 1
        n_soft   = len(run["boundaries"]) - n_hard
        print(f"Run {ri+1}/{len(runs)}  speaker={run['speaker']}  "
              f"tokens={len(run['tokens'])}  "
              f"hard_boundaries={n_hard}  soft_boundaries={n_soft}")

        run_sents = []
        for sub in sub_runs:
            if not sub["tokens"]:
                continue
            if len(sub["boundaries"]) == 0:
                # Single EXB utterance — bypass Gemini
                run_sents.append([te.token for te in sub["tokens"]])
            else:
                # Multiple utterances — ask Gemini to group semantically
                sents = segment_utterances(sub, model_name)
                run_sents.extend(sents)

        sentences_per_run.append(run_sents)
        done += len(run["tokens"])
        print(f"  -> {len(run_sents)} sentence(s)  ({done}/{total} tokens done)")

    return sentences_per_run


In [ ]:
# ----------------------------
# 5) WRITE OUTPUTS
# ----------------------------

def _all_sentences(sentences_per_run):
    """Flatten per-run sentences into one ordered list."""
    return [s for run_sents in sentences_per_run for s in run_sents]


def write_sentences_txt(sentences_per_run, out_path):
    with open(out_path, "w", encoding="utf-8") as f:
        for s in _all_sentences(sentences_per_run):
            f.write(" ".join(s) + "\n")


def write_sentences_json(sentences_per_run, out_path):
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({"sentences": _all_sentences(sentences_per_run)}, f,
                  ensure_ascii=False, indent=2)


_PAUSE_UPOS = "PUNCT"

def _pause_row(marker, gap_sec, tok_id):
    misc = f"IsPause=yes|PauseDur={gap_sec:.3f}"
    return [str(tok_id), marker, marker, _PAUSE_UPOS, "_", "_", "_", "_", "_", misc]


def write_corrected_conllup(runs, sentences_per_run, out_path, base_id):
    """
    Write re-segmented CoNLL-U with:
      - Sentence boundaries strictly within speaker runs (no cross-speaker merging).
      - Pause markers (·, ··, ···) inserted as real token rows wherever an EXB
        utterance boundary falls *inside* a merged sentence.
      - Pauses at sentence ends are NOT inserted — the break already represents them.
    """
    s_i = 0

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("# global.columns = ID FORM LEMMA UPOS XPOS FEATS HEAD DEPREL DEPS MISC\n\n")

        for run, run_sentences in zip(runs, sentences_per_run):
            run_tokens   = run["tokens"]
            boundary_map = {b.after_tok_idx: b.gap_sec for b in run["boundaries"]}
            run_offset   = 0

            for sent_tokens in run_sentences:
                s_i += 1
                sent_len = len(sent_tokens)
                span     = run_tokens[run_offset : run_offset + sent_len]

                if [te.token for te in span] != sent_tokens:
                    raise RuntimeError(
                        f"Token mismatch at sentence {s_i}: "
                        f"expected {sent_tokens}, got {[te.token for te in span]}"
                    )

                # Build items: ('tok', TokenEntry) | ('pause', (marker, gap))
                items = []
                for local_idx, te in enumerate(span):
                    global_idx = run_offset + local_idx
                    items.append(("tok", te))
                    # Internal boundary only (both sides inside this sentence)
                    if global_idx in boundary_map and local_idx < sent_len - 1:
                        gap    = boundary_map[global_idx]
                        marker = pause_marker(gap)
                        if marker:
                            items.append(("pause", (marker, gap)))

                run_offset += sent_len

                tok_entries = [d for typ, d in items if typ == "tok"]

                speakers   = {te.meta.get("speaker")           for te in tok_entries if te.meta.get("speaker")}
                abbrs      = {te.meta.get("speaker_abbr")      for te in tok_entries if te.meta.get("speaker_abbr")}
                ages       = {te.meta.get("speaker_age")       for te in tok_entries if te.meta.get("speaker_age")}
                genders    = {te.meta.get("speaker_gender")    for te in tok_entries if te.meta.get("speaker_gender")}
                educations = {te.meta.get("speaker_education") for te in tok_entries if te.meta.get("speaker_education")}
                locations  = {te.meta.get("location")          for te in tok_entries if te.meta.get("location")}

                start_times = [_safe_float(te.meta.get("start_time")) for te in tok_entries]
                end_times   = [_safe_float(te.meta.get("end_time"))   for te in tok_entries]
                st = next((x for x in start_times if x is not None), None)
                et = next((x for x in reversed(end_times) if x is not None), None)

                text_parts = [d[0] if typ == "pause" else d.token for typ, d in items]
                sent_text  = " ".join(text_parts)

                f.write(f"# sent_id = {base_id}-r{s_i:06d}\n")
                f.write(f"# text = {sent_text}\n")
                if len(speakers)   == 1: f.write(f"# speaker = {next(iter(speakers))}\n")
                if len(abbrs)      == 1: f.write(f"# speaker_abbr = {next(iter(abbrs))}\n")
                if len(ages)       == 1: f.write(f"# speaker_age = {next(iter(ages))}\n")
                if len(genders)    == 1: f.write(f"# speaker_gender = {next(iter(genders))}\n")
                if len(educations) == 1: f.write(f"# speaker_education = {next(iter(educations))}\n")
                if len(locations)  == 1: f.write(f"# location = {next(iter(locations))}\n")
                if st is not None:       f.write(f"# start_time = {st:.3f}\n")
                if et is not None:       f.write(f"# end_time = {et:.3f}\n")

                row_id = 1
                for typ, dat in items:
                    if typ == "tok":
                        cols = list(dat.cols)
                        cols[0] = str(row_id)
                        f.write("\t".join(cols) + "\n")
                    else:
                        marker, gap = dat
                        f.write("\t".join(_pause_row(marker, gap, row_id)) + "\n")
                    row_id += 1

                f.write("\n")


In [ ]:

# ----------------------------
# 6) RUN PIPELINE
# ----------------------------
uploaded = files.upload()
in_name = next(iter(uploaded.keys()))
in_path = in_name  # Colab saves to /content/<filename>

# Parse
token_entries, all_tokens = parse_conllu_like(in_path)
print(f"Loaded  : {in_path}")
print(f"Tokens  : {len(all_tokens)}")

# Split into speaker runs (never merge across speaker changes)
runs = build_speaker_runs(token_entries)
print(f"\nSpeaker runs: {len(runs)}")
for i, run in enumerate(runs):
    bcount = len(run["boundaries"])
    print(f"  Run {i+1:2d}  speaker={run['speaker']:8s}  "
          f"tokens={len(run['tokens']):3d}  "
          f"EXB-utterance-boundaries={bcount}"
          + (f"  gaps=[{', '.join(f'{b.gap_sec:.2f}s' for b in run['boundaries'])}]"
             if bcount else ""))

# Segment each run independently via Gemini
print("\nSegmenting …")
sentences_per_run = segment_all_runs(runs)

total_sents = sum(len(s) for s in sentences_per_run)
print(f"\nTotal sentences: {total_sents}")

# Output paths
base_id      = os.path.splitext(os.path.basename(in_path))[0]
out_sent_txt  = f"{base_id}.sentences.txt"
out_sent_json = f"{base_id}.sentences.json"
out_conll     = f"{base_id}.corrected.conllup"

write_sentences_txt(sentences_per_run,  out_sent_txt)
write_sentences_json(sentences_per_run, out_sent_json)
write_corrected_conllup(runs, sentences_per_run, out_conll, base_id=base_id)

print(f"\nWrote outputs:")
print(f"  {out_sent_txt}")
print(f"  {out_sent_json}")
print(f"  {out_conll}")

# Download
files.download(out_sent_txt)
files.download(out_sent_json)
files.download(out_conll)
